**Healthcare Data Objective: Assessing the Strain on Healthcare Resources**

Importing necessary libraries

In [1]:
import math
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns
import textwrap
import duckdb

%matplotlib inline

In [15]:
import os
print(os.listdir('DataAnalysis/UK_based_EDA'))



FileNotFoundError: [WinError 3] The system cannot find the path specified: 'DataAnalysis/UK_based_EDA'

In [13]:
df = duckdb.read_csv('DataAnalysis\\UK_based_EDA\\diabetes.csv')



df = duckdb.query("SELECT * FROM df").to_df()
df.head()

IOException: IO Error: No files found that match the pattern "DataAnalysis\UK_based_EDA\diabetes.csv"

In [ ]:
# Load the CSV file into a DataFrame
#file_path = 'merged_covid_data.csv'

death = pd.read_csv('COVID_DEATHS.csv')
vaccine = pd.read_csv('COVID_VACCINATIONS.csv')

Dataset Shape

In [ ]:
print(death.shape)
print(vaccine.shape)

Checking the Data types in the covid death dataset

In [ ]:
death.info()

Checking the Data types in the covid vaccine dataset

In [ ]:
vaccine.info()

Checking for duplicates in both df

In [ ]:
# Check for duplicates in 'death' DataFrame
duplicate_rows_death = death.duplicated()
print("Number of duplicate rows in 'death':", duplicate_rows_death.sum())

# Check for duplicates in 'vaccine' DataFrame
duplicate_rows_vaccine = vaccine.duplicated()
print("Number of duplicate rows in 'vaccine':", duplicate_rows_vaccine.sum())

Combining the dfs  into one df

In [ ]:
combined = pd.merge(death, vaccine, how='outer')
print(combined.shape)
combined.head()


**Cleaning andPreprocessing**

In [ ]:
#Finding out anomalies in the dataframe by getting statistical summary of each column
combined.describe()

In [ ]:
# Remove rows where 'continent' is null
combined = combined.dropna(subset=['continent'])
print(combined.shape)

In [ ]:
#Converting Date from Object to Date Type
combined['date'] = pd.to_datetime(combined['date'])

In [ ]:
# Filling NaN Values with 0
combined.fillna(0, inplace=True)
combined.head()

In [ ]:
combined.shape

In [ ]:
# Check for null values across all columns
null_columns = combined.columns[combined.isnull().any()].tolist()
print(f"Columns with null values: {null_columns}")

In [ ]:
#Checking for Future Dates
from datetime import datetime
today = datetime.now()

anomalies_future_dates = combined[pd.to_datetime(combined['date']) > today]
anomalies_future_dates.shape

In [ ]:
#Removing columns that might be irrelevant for most analyses
# List of columns to remove
columns_to_remove = ['iso_code', 'new_cases_smoothed', 'new_deaths_smoothed', 'total_cases_per_million', 
                     'new_cases_per_million']

# Remove specified columns
combined_dropped = combined.drop(columns=columns_to_remove)
combined_dropped

In [ ]:
combined_dropped

In [ ]:
# Correlation matrix for selected variables
correlation_matrix = combined_dropped[['total_cases', 'total_deaths', 'icu_patients', 'total_vaccinations', 'stringency_index']].corr()

# Plot the correlation matrix using seaborn
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

**Data Analytics**

In [ ]:
regional_df= combined_dropped.copy()
regional_df.head()

***Correlation Analyses***

In [ ]:
selected_columns = [
    'total_cases', 'total_deaths', 'total_tests', 'positive_rate',
    'total_vaccinations', 'population_density', 'hospital_beds_per_thousand'
]
correlation_matrix2 = regional_df[selected_columns].corr() # Compute the correlation matrix

sns.heatmap(correlation_matrix2, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

**KPIs Calculation**

1. Grouping the dataframe with continent location year month and sorting the values by continent and reseting the index

2.  Calculate required KPIs as added columns

In [ ]:
# Extract year and month from the 'date' column to new columns
regional_df['year'] = regional_df['date'].dt.year
regional_df['month'] = regional_df['date'].dt.strftime('%b')

# Group the dataframe by specified columns and sum the numeric columns
grouped_df = regional_df.groupby(['continent', 'location', 'year', 'month']).sum(numeric_only=True).reset_index()

# Calculate new columns based on the grouped data
grouped_df['percent_of_new_cases'] = (grouped_df['new_cases'] / grouped_df['total_cases'].replace(0, np.nan)) * 100
grouped_df['vaccination_success_rate'] = (grouped_df['people_fully_vaccinated'] / grouped_df['population'].replace(0, np.nan)) * 100
grouped_df['case_fatality_rate'] = (grouped_df['total_deaths'] / grouped_df['total_cases'].replace(0, np.nan)) * 100
grouped_df['vaccination_coverage'] = (grouped_df['total_vaccinations'] / grouped_df['population'].replace(0, np.nan)) * 100
grouped_df['healthcare_strain_index'] = grouped_df['total_cases'] / grouped_df['hospital_beds_per_thousand'].replace(0, np.nan)

grouped_df.head()

In [ ]:
# Fill NaN values with 0 in the new columns
new_columns = ['percent_of_new_cases', 'vaccination_success_rate', 'case_fatality_rate', 'vaccination_coverage', 'healthcare_strain_index']
grouped_df[new_columns] = grouped_df[new_columns].fillna(0)

# Convert columns to categorical types for efficiency
cat_columns = ['continent', 'location', 'year', 'month']
grouped_df[cat_columns] = grouped_df[cat_columns].astype('category')

#print(regional_df['healthcare_strain_index'].describe())

***Regional Time-Series Analysis for Global Trends***

***Bar Plots for KPIs by Continent***

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 18))
axes = axes.flatten()

# Case Fatality Rate by Continent
sns.barplot(x='continent', y='case_fatality_rate', data=grouped_df, ax=axes[0], errorbar=None)
axes[0].set_title('Case Fatality Rate by Continent')

# Test Positivity Rate by Continent
sns.barplot(x='continent', y='positive_rate', data=grouped_df, ax=axes[1], errorbar=None)
axes[1].set_title('Test Positivity Rate by Continent')

# Vaccination Coverage by Continent
sns.barplot(x='continent', y='vaccination_coverage', data=grouped_df, ax=axes[2], errorbar=None)
axes[2].set_title('Vaccination Coverage by Continent')

# Healthcare Strain Index by Continent
sns.barplot(x='continent', y='healthcare_strain_index', data=grouped_df, ax=axes[3], errorbar=None)
axes[3].set_title('Healthcare Strain Index by Continent')

# Percentage of New cases by Continent
sns.barplot(x='continent', y='percent_of_new_cases', data=grouped_df, ax=axes[4], errorbar=None)
axes[4].set_title('Percentage of New Cases by Continent')

# Vaccination Success Rate  by Continent
sns.barplot(x='continent', y='vaccination_success_rate', data=grouped_df, ax=axes[5], errorbar=None)
axes[5].set_title('Vaccination Success Rate by Continent')

plt.tight_layout()
plt.show()


***Trend Analysis*** for 9 most relevant trends 

In [ ]:
columns_to_plot = [
    'total_cases',  'total_vaccinations','total_deaths', 
  'reproduction_rate', 'total_tests', 'positive_rate',
   'population',  'hosp_patients','icu_patients'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(columns_to_plot):
    sns.lineplot(x='year', y=col, data=grouped_df, ax=axes[i])
    axes[i].set_title(f'{col} Over the Years')

plt.tight_layout()
plt.show()

***Trends and Relationships Across Different Continents Over the Years.***

In [ ]:
plot_specs = [# Columns and plot types
    ('year', 'line', 'Total Cases Over the Years by Continent', 'total_cases'),
    ('year', 'line', 'Total Deaths Over the Years by Continent', 'total_deaths'),
    ('year', 'line', 'Vaccination Success Rate Over the Years by Continent', 'vaccination_success_rate'),
    ('year', 'line', 'Percent of New Cases Over the Years by Continent', 'percent_of_new_cases'),
    ('total_cases', 'scatter', 'Total Cases vs Total Deaths by Continent', 'total_deaths'),
    ('total_vaccinations', 'scatter', 'Total Vaccinations vs Population by Continent', 'population')
]

fig, axes = plt.subplots(len(plot_specs), 1, figsize=(20, 40)) # Create subplots

# Flatten axes for easy iteration if more than 1 row and 1 column
if len(plot_specs) > 1:
    axes = axes.flatten()

# Loop through plot specifications
for ax, (x, plot_type, title, y) in zip(axes, plot_specs):
    if plot_type == 'line':
        sns.lineplot(x=x, y=y, hue='continent', data=grouped_df, ax=ax)
    elif plot_type == 'scatter':
        sns.scatterplot(x=x, y=y, hue='continent', data=grouped_df, ax=ax)
           
    ax.set_title(title)
    ax.set_xlabel(x.replace('_', ' ').title())
    ax.set_ylabel(y.replace('_', ' ').title())

# Adjust layout
plt.tight_layout()
plt.show()


***Top 10 Locations Analysis Over the Past 5 Years***

 Identifying Top 10 Locations

In [ ]:
location_df = combined_dropped.sort_values(['date', 'location'])

In [ ]:
# Extract year and month from the 'date' column to new columns
location_df['year'] = location_df['date'].dt.year
location_df['month'] = location_df['date'].dt.strftime('%b')

Plotting Important Metrics for Top 10 Locations

In [ ]:
location_df['percent_of_new_cases'] = (location_df['new_cases'] / location_df['total_cases'].replace(0, np.nan)) * 100
location_df['vaccination_success_rate'] = (location_df['people_fully_vaccinated'] / location_df['population'].replace(0, np.nan)) * 100
location_df.fillna(0, inplace=True)

columns_to_plot = ['total_cases', 'total_deaths', 'vaccination_success_rate', 'percent_of_new_cases', 'total_boosters']
last_5_years = location_df['year'].unique()[-5:] # Take the last 5 unique years

In [ ]:
# Precompute the top 10 locations for each column for each year
top_locations = {}

for year in last_5_years:
    top_locations[year] = {}
    yearly_data = location_df[location_df['year'] == year]
    
    for col in columns_to_plot:
        # For cumulative columns, we take the last available data point in the year for each location
        if col != 'positive_rate':
            sorted_df = yearly_data.drop_duplicates(subset=['location'], keep='last').sort_values(by=col, ascending=False)
        # For non-cumulative columns like positive_rate, we can directly sort
        else:
            sorted_df = yearly_data.sort_values(by=col, ascending=False)
        
        top_10_locations = sorted_df['location'].head(10).values
        top_locations[year][col] = top_10_locations

In [ ]:
# Plotting

fig, axes = plt.subplots(len(columns_to_plot), len(last_5_years), figsize=(40, 30))

for j, col in enumerate(columns_to_plot):
    for i, year in enumerate(last_5_years):
        ax = axes[j, i]
        top_10_for_col = top_locations[year][col]
        data_to_plot = location_df[(location_df['year'] == year) & (location_df['location'].isin(top_10_for_col))]

        # Check the type of plot
        if col in ['total_cases', 'total_deaths', 'total_boosters']:  # Barplot
            sns.barplot(x='location', y=col, data=data_to_plot, ax=ax, order=top_10_for_col, errorbar=None)
        else:  # Lineplot
            sns.lineplot(x='location', y=col, data=data_to_plot, ax=ax, sort=False)

        ax.set_title(f'Top 10 locations for {col} in {year}')
        
        # Explicitly set tick positions to match the number of labels
        ax.set_xticks(range(len(top_10_for_col)))
        ax.set_xticklabels([textwrap.fill(label.get_text(), 10) for label in ax.get_xticklabels()], rotation=0)

plt.tight_layout()
plt.show()

***MoM deaths and cases for the past 5 years***

In [ ]:
mom_df = location_df.copy()
mom_df['week'] = mom_df['date'].dt.isocalendar().week

In [ ]:
import calendar

# Create an ordered list of abbreviated month names
mom_df['month'] = pd.Categorical(mom_df['month'], categories=list(calendar.month_abbr)[1:], ordered=True)

mom_df = mom_df.sort_values(['year', 'month', 'date'])# Sort DataFrame by year and month
mom_df

In [ ]:
filtered_df = mom_df[mom_df['year'].isin(last_5_years)]

# Group by 'year' and 'month', and sum 'total_cases' and 'total_deaths'
grouped_df = filtered_df.groupby(['year', 'month']).agg({'total_cases': 'sum', 'total_deaths': 'sum'}).reset_index()

# Calculate the log; replace negative infinity and NaN with a small number
grouped_df['log_total_cases'] = np.log(grouped_df['total_cases'] + 1e-10)
grouped_df['log_total_deaths'] = np.log(grouped_df['total_deaths'] + 1e-10)

 Plotting MoM and WoW Metrics

In [ ]:
fig, axes = plt.subplots(len(last_5_years), 1, figsize=(15, 7 * len(last_5_years)))

# Loop through each year
for i, year in enumerate(last_5_years):
    ax = axes[i]
    yearly_data = grouped_df[grouped_df['year'] == year].copy()  # Create a copy of the DataFrame

    # Exclude negative points from log values using .loc accessor
    yearly_data.loc[yearly_data['log_total_cases'] <= 0, 'log_total_cases'] = 0
    yearly_data.loc[yearly_data['log_total_deaths'] <= 0, 'log_total_deaths'] = 0

    x = np.arange(len(yearly_data['month']))  # the label locations
    width = 0.35  # the width of the bars

    # Create clustered bar chart
    ax.bar(x - width/2, yearly_data['log_total_cases'], width, label='Log of Total Cases')
    ax.bar(x + width/2, yearly_data['log_total_deaths'], width, label='Log of Total Deaths')

    # Add labels and title
    ax.set_xlabel('Month')
    ax.set_ylabel('Log Scale Value (Excluding Negative Points)')
    ax.set_title(f'Log of Monthly Sum of Total Cases and Deaths for {year}')
    ax.set_xticks(x)
    ax.set_xticklabels(yearly_data['month'])
    ax.legend()

plt.tight_layout()
plt.show()


***Geographical Analysis***

In [ ]:
grouped_df

In [ ]:
# Plotting population density vs total cases
sns.scatterplot(x='population_density', y='total_cases', data=regional_df)
plt.title('Population Density vs Total Cases')
plt.show()

# Plotting median age vs total cases
sns.scatterplot(x='median_age', y='total_cases', data=regional_df)
plt.title('Median Age vs Total Cases')
plt.show()


# Assuming you have a DataFrame 'df' with 'Latitude', 'Longitude', and 'Confirmed Cases'
import folium

m = folium.Map(location=[20, 0], zoom_start=3)

for index, row in df.iterrows():
    folium.CircleMarker([row['Latitude'], row['Longitude']],
                        radius=row['Confirmed Cases']/1000000,
                        color="red",
                        fill=True,
                        fill_opacity=0.4,
                        ).add_to(m)
                        
m.save("Geographical_Analysis.html")
s

***Economic/Policy Analysis***

In [ ]:
# Plotting GDP per capita vs total cases
sns.scatterplot(x='gdp_per_capita', y='total_cases', data=regional_df)
plt.title('GDP Per Capita vs Total Cases')
plt.show()

# Plotting stringency index vs total cases
sns.scatterplot(x='stringency_index', y='total_cases', data=regional_df)
plt.title('Stringency Index vs Total Cases')
plt.show()

# Assuming 'df_policy' DataFrame contains 'Date', 'Policy_Type', and 'Effectiveness_Score'
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.lineplot(x='Date', y='Effectiveness_Score', hue='Policy_Type', data=df_policy)
plt.title('Effectiveness of Different Policies Over Time')
plt.xlabel('Date')
plt.ylabel('Effectiveness Score')
plt.show()

